# ConsentML + Snowflake: sourcing training data and storing lineage

[ConsentML](https://github.com/KaranamLokesh/consentml) integrates with Snowflake in **two independent places**, and this notebook walks through both:

1. **`SnowflakeSource`** — read the training set *out of* Snowflake and record its provenance, exactly like `PostgresSource`. It plugs straight into the `@track` decorator; lineage still lands in the default SQLite store.
2. **`SnowflakeLineageStore`** — persist the hash-chained audit log *in* Snowflake instead of SQLite. This backend is **Python-API only** in v1 (the CLI and `@track` stay SQLite-only), so you construct it directly via `open_store({...})` and drive it with the store's own methods.

The two share nothing but a connection library — pick whichever you need.

> **This notebook needs a real Snowflake account.** Unlike [`consentml_demo.ipynb`](consentml_demo.ipynb), there is no local double: the cells connect to Snowflake, so they only run once you fill in credentials below. Read it top-to-bottom even without an account — every cell is annotated.

## 1. Setup

Install ConsentML with the Snowflake extra (pulls in `snowflake-connector-python`). Works locally and in Google Colab.

> **Version note.** Snowflake support landed *after* the 0.1.1 PyPI release. Until the next release, install from a source checkout (`pip install -e '.[snowflake]'` in a clone of the repo) — installing `consentml[snowflake]` from PyPI fetches 0.1.1, which does not yet include `SnowflakeSource` / `SnowflakeLineageStore`.

In [ ]:
# Locally, this upgrades your existing install to include the Snowflake driver.
# In Colab it installs from PyPI.
%pip install -q 'consentml[snowflake]'

import consentml
print("consentml", consentml.__version__, "ready")

## 2. Connection

Snowflake credentials come from you, not from a config file checked into the repo. Fill these in (or wire them to `os.environ` / Colab secrets). ConsentML supports **user/password** and **key-pair** auth; this notebook uses user/password.

Two things worth knowing before you connect:

- **No read-only enforcement.** Unlike `PostgresSource`, Snowflake exposes no connection-level read-only flag, so ConsentML can't force it. The `role` you pass in should hold **read-only grants** on the data you read.
- **Credentials never reach provenance.** Only `account`/`database`/`schema`/`warehouse` are recorded — never `user`, `password`, or a key.

In [ ]:
import getpass
import os

# Credentials come from you at runtime -- never hardcode them in a notebook
# that ships in a public repo. The non-secret coordinates default to
# placeholders you can override with environment variables; the password is
# read from SNOWFLAKE_PASSWORD or prompted, so it never lives in this file.
connection = {
    "account": os.environ.get("SNOWFLAKE_ACCOUNT", "your_account"),      # e.g. ab12345.us-east-1
    "user": os.environ.get("SNOWFLAKE_USER", "your_user"),               # ideally a read-only role
    "database": os.environ.get("SNOWFLAKE_DATABASE", "your_database"),
    "schema": os.environ.get("SNOWFLAKE_SCHEMA", "PUBLIC"),
    "warehouse": os.environ.get("SNOWFLAKE_WAREHOUSE", "your_warehouse"),
    # "role": os.environ.get("SNOWFLAKE_ROLE", "ANALYST_READONLY"),  # read-only grants recommended
}

# Password auth (shown here). For a programmatic access token (PAT) instead,
# drop the password line and use:
#     connection["authenticator"] = "PROGRAMMATIC_ACCESS_TOKEN"
#     connection["token"] = os.environ["SNOWFLAKE_TOKEN"]
connection["password"] = (
    os.environ.get("SNOWFLAKE_PASSWORD") or getpass.getpass("Snowflake password: ")
)

## 3. `SnowflakeSource`: Snowflake supplies the training data

`@track` loads the data from Snowflake, passes it to your training function, and records the run. You don't hand `train()` any data — ConsentML does, so the lineage can't drift from what the model actually trained on.

Here lineage is written to the **default SQLite store** (`lineage.db`); only the *data source* is Snowflake.

In [ ]:
from consentml import track
from consentml.sources.snowflake import SnowflakeSource


@track(
    model_name="readmission-risk",
    source=SnowflakeSource(
        connection=connection,
        query="SELECT patient_id, age, ldl, outcome FROM patients",
        subject_id_col="patient_id",
    ),
)
def train(df):
    # A real trainer fits a model here; ConsentML only needs the return value,
    # which it hashes (SHA-256 of the pickle) into the lineage record.
    return {"n_rows": len(df), "columns": list(df.columns)}


model = train()   # no argument: ConsentML runs the query and supplies the data
model

### What got recorded

The provenance holds the exact query, its SHA-256, the tables the plan touched (best-effort via `EXPLAIN USING JSON`), and where the data came from — never the credentials. `referenced_tables_source` is `"explain"` when Snowflake's plan was parseable, `"unavailable"` otherwise (advisory only — a failed `EXPLAIN` never fails the training run).

In [ ]:
import json
from consentml import verify_audit_log

report = verify_audit_log(db_path="lineage.db")
print("audit log ok:", report.ok)

# The stored provenance for the run we just recorded:
from consentml.store import open_store

store = open_store(db_path="lineage.db")
try:
    run = store.latest_run_for_model("readmission-risk")
finally:
    store.close()
json.loads(run["provenance"])

## 4. `SnowflakeLineageStore`: the audit log lives in Snowflake

Now the second, independent integration: keep the **hash-chained lineage itself** in Snowflake. This backend is reached through `open_store({...})` — a **dict** target selects Snowflake, a path/`None` stays SQLite. (A `snowflake://` URI is deliberately rejected: it can't safely carry a password, so pass a dict.)

`@track` can't route here in v1, so you record and read runs with the store's own interface methods. The schema is denormalized relative to SQLite (no subject interning — a columnar warehouse doesn't benefit from it), but the audit-log row shape and the `entry_hash` formula are **byte-for-byte identical**, so the same code verifies a chain no matter which backend wrote it.

In [ ]:
import hashlib
import pickle
from datetime import datetime, timezone

from consentml.hashing import hash_subject_id
from consentml.store import open_store

# A dict target -> SnowflakeLineageStore (creates its tables on first open).
sf_store = open_store(connection)

subject_ids = ["patient-001", "patient-002", "patient-003"]
demo_model = {"kind": "demo"}
now = datetime.now(timezone.utc).isoformat()

run_id = sf_store.record_training_run(
    model_name="readmission-risk",
    model_hash=hashlib.sha256(pickle.dumps(demo_model)).hexdigest(),
    provenance={"kind": "demo", "note": "manual store example"},
    subject_ids_hashed=True,
    subject_id_values=[hash_subject_id(s) for s in subject_ids],
    started_at=now,
    finished_at=now,
)
print("recorded run", run_id, "in Snowflake")

### Read it back

Every `LineageStore` method works the same over Snowflake as over SQLite — look up which runs a subject appears in, and walk the hash-chained audit log. (Subject IDs were hashed on write, so look them up by the same hash.)

In [ ]:
runs = sf_store.runs_for_subject_value(hash_subject_id("patient-001"))
print("patient-001 appears in", len(runs), "run(s)")

for entry in sf_store.audit_entries():
    print(f"  audit #{entry['id']}: {entry['event_type']} "
          f"({entry['entry_hash'][:12]}...)")

sf_store.close()

> **Concurrency note.** The Snowflake store assumes a **single logical writer** per lineage table. The audit chain is a read-then-append (read the last `entry_hash`, write a row whose `prev_hash` is that value); a warehouse won't serialize that for you, so two concurrent writers could fork the chain. Coordinating multiple writers is an explicit non-goal in v1.

## Wrap-up

- **`SnowflakeSource`** — Snowflake as the training-data source, provenance recorded, works with `@track`. Lineage store is your choice (SQLite by default).
- **`SnowflakeLineageStore`** — the tamper-evident audit log persisted in Snowflake, via the Python API.

Both are opt-in; SQLite stays the default and every offline user, test, and database is untouched. See the [README](https://github.com/KaranamLokesh/consentml#readme) for key-pair auth and the full `LineageStore` interface.